# 01 · Learning the kinematics of a 2-link arm  *(warm-up, ~20 min)*

Before the real robot, a **toy problem** we can fully see on a 2D plot:

1. generate data from a 2-link planar arm (we play the role of the robot);
2. learn the **forward** kinematics $x = f(q)$ with a neural network in PyTorch;
3. try to learn the **inverse** kinematics $q = g(x)$ the same way... and watch it **fail**;
4. fix it with the **distal teacher** idea.

Everything you see here is exactly what we will do on the 7-joint Franka, only smaller.

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
L1, L2 = 1.0, 0.8

def fk(q):
    """TRUE forward kinematics. q: (N, 2) -> tip position (N, 2). Works for numpy and torch."""
    lib = torch if torch.is_tensor(q) else np
    q1, q2 = q[:, 0], q[:, 1]
    x = L1 * lib.cos(q1) + L2 * lib.cos(q1 + q2)
    y = L1 * lib.sin(q1) + L2 * lib.sin(q1 + q2)
    return lib.stack([x, y], 1)

def draw_arm(q, ax, **style):
    """Draw the arm in configuration q = (q1, q2)."""
    elbow = np.array([L1 * np.cos(q[0]), L1 * np.sin(q[0])])
    tip = fk(np.array([q]))[0]
    ax.plot([0, elbow[0], tip[0]], [0, elbow[1], tip[1]], "-o", **style)

## 1. The dataset
Random configurations ("motor babbling") and the tip positions they produce. To keep the picture simple, joint 1 moves in $[-\pi/2, \pi/2]$ and joint 2 in $[-2.5, 2.5]$ rad: **both elbow-up and elbow-down** postures appear.

In [ ]:
N = 20000
q_data = np.stack([rng.uniform(-np.pi / 2, np.pi / 2, N), rng.uniform(-2.5, 2.5, N)], 1)
x_data = fk(q_data)

q_t = torch.tensor(q_data, dtype=torch.float32)   # PyTorch works with "tensors" (= numpy arrays + gradients)
x_t = torch.tensor(x_data, dtype=torch.float32)
tr, te = slice(0, 16000), slice(16000, N)          # 80% to learn (tr), 20% to test (te)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(q_data[:3000, 0], q_data[:3000, 1], s=1); ax[0].set(xlabel="q1 [rad]", ylabel="q2 [rad]", title="joint space (inputs)")
ax[1].scatter(x_data[:3000, 0], x_data[:3000, 1], s=1, c="tab:orange"); ax[1].set(xlabel="x [m]", ylabel="y [m]", title="task space (labels)")
ax[1].axis("equal"); plt.tight_layout(); plt.show()

## 2. Autograd in 30 seconds
PyTorch can compute **derivatives automatically**. This is what makes training (and later our controller) possible.

Example: $L(w) = (3w - 6)^2$. By hand, $dL/dw = 6(3w-6)$, so at $w = 1$ we expect $-18$.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)   # "please track operations on w"
L = (3 * w - 6) ** 2
L.backward()                                # compute dL/dw
print("L =", L.item(), "  dL/dw =", w.grad.item())

# Gradient descent by hand: move w against the gradient, a little at a time
w = torch.tensor(1.0, requires_grad=True)
for step in range(20):
    L = (3 * w - 6) ** 2
    L.backward()
    with torch.no_grad():
        w -= 0.02 * w.grad      # learning rate = 0.02
        w.grad.zero_()
print("after 20 steps w =", round(w.item(), 4), "(the minimum is at w = 2)")

## 3. Forward kinematics with an MLP
A network with 2 inputs, two hidden layers of 64 neurons, 2 outputs. The **training loop** has always the same 4 lines: *predict → loss → backward → step*.

In [ ]:
def make_mlp(n_in, n_out, hidden=64):
    return nn.Sequential(nn.Linear(n_in, hidden), nn.SiLU(),
                         nn.Linear(hidden, hidden), nn.SiLU(),
                         nn.Linear(hidden, n_out))

def train(model, loss_fn, inputs, targets, epochs=30, lr=3e-3, batch=128):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        perm = torch.randperm(len(inputs))                    # shuffle every epoch
        for i in range(0, len(inputs), batch):
            idx = perm[i:i + batch]                            # one mini-batch
            loss = loss_fn(model(inputs[idx]), targets[idx])   # 1. predict and measure the error
            opt.zero_grad()                                    # 2. reset old gradients
            loss.backward()                                    # 3. backpropagation
            opt.step()                                         # 4. update the weights
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:3d}  loss {loss.item():.6f}")
    return model

mse = nn.MSELoss()
fk_net = train(make_mlp(2, 2), mse, q_t[tr], x_t[tr])

In [ ]:
with torch.no_grad():
    err = (fk_net(q_t[te]) - x_t[te]).norm(dim=1)
print(f"FK test error: mean {1000 * err.mean():.1f} mm, max {1000 * err.max():.1f} mm  (arm reach: 1800 mm)")

## 4. Inverse kinematics, the naive way
Same data, **inputs and labels swapped**: learn $q = g(x)$ by minimizing $\|g(x) - q\|^2$.
Then we check the result honestly: we put the predicted $q$ into the **true** FK. Does the arm reach $x$?

In [ ]:
ik_naive = train(make_mlp(2, 2), mse, x_t[tr], q_t[tr])
with torch.no_grad():
    reached = fk(ik_naive(x_t[te]))
err_naive = (reached - x_t[te]).norm(dim=1)
print(f"naive IK reaching error: mean {1000 * err_naive.mean():.0f} mm")

### Why? Look at one target
For a target $x^*$, collect the dataset samples whose tip lands near $x^*$, and look at their configurations.

In [ ]:
x_star = np.array([1.0, 0.8])
near = np.linalg.norm(x_data - x_star, axis=1) < 0.03
q_near = q_data[near]
q_avg = q_near.mean(0)
with torch.no_grad():
    q_pred = ik_naive(torch.tensor([x_star], dtype=torch.float32))[0].numpy()

fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
ax[0].scatter(q_near[:, 0], q_near[:, 1], s=8, label="dataset: q that reach x*")
ax[0].plot(*q_avg, "r*", ms=15, label="their average")
ax[0].plot(*q_pred, "kx", ms=12, mew=3, label="naive network")
ax[0].set(xlabel="q1", ylabel="q2", title="joint space"); ax[0].legend()
for q in q_near[::max(1, len(q_near) // 6)]:
    draw_arm(q, ax[1], color="tab:blue", alpha=0.4)
draw_arm(q_avg, ax[1], color="red", lw=3)
ax[1].plot(*x_star, "g*", ms=18, label="target x*")
ax[1].set(title="task space: two families of solutions; their average misses"); ax[1].axis("equal"); ax[1].legend()
plt.tight_layout(); plt.show()

**The network did its job perfectly — on the wrong question.** With the MSE loss, the best prediction for an input is
the **average** of all the labels seen for that input. Here the labels form two separate groups (elbow up, elbow down)
and their average is an *elbow-straight* configuration that does not reach the target.

> The set of solutions is **not convex**: the average of two solutions is not a solution.

## 5. The distal teacher (Jordan & Rumelhart, 1992)
Do not tell the network *which* $q$ to output. Ask only that the $q$ it outputs **reaches** $x$, measured through
the **learned forward model** (frozen):

$$\min_{\phi}\; \big\|\, f_\theta\big(g_\phi(x)\big) - x \,\big\|^2$$

The error is measured in task space ("distal" = far from the network output) and backpropagated *through* $f_\theta$.
Note: **no joint labels are needed**, only target positions!

In [ ]:
for p in fk_net.parameters():
    p.requires_grad_(False)                       # freeze the forward model

def distal_loss(q_pred, x_target):
    return ((fk_net(q_pred) - x_target) ** 2).mean()

ik_distal = train(make_mlp(2, 2), distal_loss, x_t[tr], x_t[tr])   # label = x itself!
with torch.no_grad():
    err_distal = (fk(ik_distal(x_t[te])) - x_t[te]).norm(dim=1)
print(f"naive IK:  mean reaching error {1000 * err_naive.mean():6.0f} mm")
print(f"distal IK: mean reaching error {1000 * err_distal.mean():6.1f} mm")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
with torch.no_grad():
    q_d = ik_distal(torch.tensor([x_star], dtype=torch.float32))[0].numpy()
draw_arm(q_pred, ax, color="red", lw=2, label="naive")
draw_arm(q_d, ax, color="tab:green", lw=3, label="distal teacher")
ax.plot(*x_star, "k*", ms=16, label="target")
ax.axis("equal"); ax.legend(); ax.set_title("The distal teacher picks ONE solution"); plt.show()

## ✍️ Optional exercises
1. Train the naive IK only on data with $q_2 > 0$ (one elbow family). Does it work now? Why is this not a general solution for a 7-joint robot?
2. Replace `nn.SiLU()` with `nn.ReLU()` in `make_mlp`. Compare FK errors.
3. Try `lr=1e-1` and `lr=1e-5` in `train`. What happens to the loss?

**Going further:** Bishop (1994) *Mixture Density Networks* learns *all* the solutions instead of one; IKFlow (Ames et al., 2022) does it with normalizing flows.